# Portfolio Optimization & Analysis
This notebook computes covariance matrices, implements quadratic programming for Minimum Variance, solves Risk Parity (Equal Risk Contribution), and visualizes the simulated Markowitz Efficient Frontier.

In [ ]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

current_path = pathlib.Path('.').resolve()
project_root = current_path if (current_path / 'src').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import returns, optimization, portfolio

sns.set_theme(style='whitegrid', palette='tab10')

prices_path = project_root / 'data' / 'processed' / 'prices_clean.csv'
prices_df = pd.read_csv(prices_path, index_col=0, parse_dates=True)

simple_rets = returns.compute_simple_returns(prices_df)
log_rets = returns.compute_log_returns(prices_df)
print(f"Loaded {len(prices_df)} prices across {len(prices_df.columns)} assets.")

## 1. Annualized Covariance Matrix

In [ ]:
cov_matrix = optimization.compute_covariance_matrix(log_rets)
ann_cov = cov_matrix * 252

plt.figure(figsize=(11, 9))
sns.heatmap(ann_cov, annot=True, fmt=".4f", cmap="mako", linewidths=0.5)
plt.title("Annualized Covariance Matrix (Sigma * 252)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Strategy Weight Allocations

In [ ]:
tickers = prices_df.columns
n_assets = len(tickers)

# Solvers
w_eq = portfolio.equal_weight(n_assets)
w_mv = optimization.minimum_variance_weights(cov_matrix)
w_rp = optimization.risk_parity_weights(cov_matrix)

weights_comp = pd.DataFrame({
    'Equal Weight': w_eq,
    'Min Variance': w_mv,
    'Risk Parity': w_rp
}, index=tickers)

print("--- Strategy Weight Allocations ---")
print((weights_comp * 100).round(2).to_string())

plt.figure(figsize=(14, 6))
weights_comp.plot(kind='bar', figsize=(14, 6), width=0.8)
plt.title("Portfolio Allocation Weights by Strategy", fontsize=14, fontweight='bold')
plt.xlabel("Ticker")
plt.ylabel("Portfolio Weight")
plt.xticks(rotation=0)
plt.legend(title="Strategy")
plt.tight_layout()
plt.show()

## 3. Marginal & Total Risk Contribution Analysis

In [ ]:
rc_eq = optimization.compute_risk_contributions(w_eq, cov_matrix) * np.sqrt(252)
rc_mv = optimization.compute_risk_contributions(w_mv, cov_matrix) * np.sqrt(252)
rc_rp = optimization.compute_risk_contributions(w_rp, cov_matrix) * np.sqrt(252)

rc_df = pd.DataFrame({
    'Equal Weight RC': rc_eq,
    'Min Variance RC': rc_mv,
    'Risk Parity RC': rc_rp
}, index=tickers)

plt.figure(figsize=(14, 6))
rc_df.plot(kind='bar', figsize=(14, 6), width=0.8, colormap='Accent')
plt.title("Annualized Risk Contributions per Asset (ERC Equalizes All Bars)", fontsize=14, fontweight='bold')
plt.xlabel("Ticker")
plt.ylabel("Risk Contribution (Annualized Vol)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Expected Return vs Volatility & Efficient Frontier Simulation

In [ ]:
mean_rets = log_rets.mean() * 252
n_simulations = 5000
sim_vols = []
sim_rets = []

np.random.seed(42)
for _ in range(n_simulations):
    raw_w = np.random.exponential(scale=1.0, size=n_assets)
    w = raw_w / np.sum(raw_w)
    ret = np.sum(w * mean_rets)
    vol = np.sqrt(w.T @ ann_cov @ w)
    sim_rets.append(ret)
    sim_vols.append(vol)

# Strategy coordinates
vol_eq = np.sqrt(w_eq.T @ ann_cov @ w_eq)
ret_eq = np.sum(w_eq * mean_rets)

vol_mv = np.sqrt(w_mv.T @ ann_cov @ w_mv)
ret_mv = np.sum(w_mv * mean_rets)

vol_rp = np.sqrt(w_rp.T @ ann_cov @ w_rp)
ret_rp = np.sum(w_rp * mean_rets)

plt.figure(figsize=(12, 7))
plt.scatter(sim_vols, sim_rets, c=np.array(sim_rets)/np.array(sim_vols), cmap='viridis', marker='o', s=8, alpha=0.5, label='Random Long-Only Portfolios')
plt.colorbar(label='Sharpe Ratio (Rf=0)')

plt.scatter(vol_eq, ret_eq, color='red', marker='X', s=200, label='Equal Weight (1/N)', zorder=5)
plt.scatter(vol_mv, ret_mv, color='blue', marker='*', s=250, label='Minimum Variance', zorder=5)
plt.scatter(vol_rp, ret_rp, color='orange', marker='D', s=180, label='Risk Parity (ERC)', zorder=5)

plt.title("Simulated Markowitz Efficient Frontier & Strategy Positions", fontsize=14, fontweight='bold')
plt.xlabel("Annualized Volatility", fontsize=11)
plt.ylabel("Annualized Expected Return", fontsize=11)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()